# 00. Which encoders can read our texts?

The assignment asks us to check that a model really supports Ukrainian before training it. The first
thing that breaks for an unsupported language is the tokenizer: words fall apart into single letters or
bytes, or become `[UNK]`. A model can't learn place names it can't even see. So before spending GPU time
we tokenize the same texts with every candidate and compare:

- **tokens per word** (fertility): about 1.3-2 is normal for a language the model was trained on;
  much higher means the words are spelled out letter by letter;
- **unknown tokens**: words the model sees as `[UNK]`, so it can't tell them apart;
- **Ukrainian letters**: whether і, ї, є, ґ have their own tokens;
- **post length in tokens**: how many test posts need more than one 256-token window.

Texts: 2,000 Ukrainian and 2,000 Russian training sentences, the 477 test posts and the place strings of
the pseudo-labeled test posts. This notebook needs no GPU.

In [1]:
import os
import sys
from pathlib import Path

if Path.cwd().name == "transformers":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import re

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

from ner.data import load_sentences, load_test_posts
from ner.encoding import model_text

pd.set_option("display.max_colwidth", 80)

CANDIDATES = {
    "mBERT": "google-bert/bert-base-multilingual-cased",
    "XLM-R base / large": "FacebookAI/xlm-roberta-base",
    "mDeBERTa-v3": "microsoft/mdeberta-v3-base",
    "EuroBERT": "EuroBERT/EuroBERT-210m",
    "ruRoBERTa-large": "ai-forever/ruRoberta-large",
    "ukr-RoBERTa": "youscan/ukr-roberta-base",
}

In [2]:
sentences = load_sentences()
uk = sentences[sentences.lang == "uk"].sample(2000, random_state=42).text.tolist()
ru = sentences[sentences.lang == "ru"].sample(2000, random_state=42).text.tolist()
test = load_test_posts()
posts = [model_text(t) for t in test.text]
places = sorted({p for row in test.locations for p in row})
corpora = {"uk sentences": uk, "ru sentences": ru, "test posts": posts, "test places": places}
WORD = re.compile(r"\w+")

In [3]:
def show_pieces(tok, text):
    """Tokens as readable text ("|" between them); byte-level BPE tokens are decoded one by one."""
    return "|".join(tok.convert_tokens_to_string([t]).strip() or "▁" for t in tok.tokenize(text))


def unk_id(tok):
    return tok.unk_token_id if tok.unk_token is not None else None


def profile(tok):
    row = {"vocab": tok.vocab_size}
    for name, texts in corpora.items():
        words = [w for t in texts for w in WORD.findall(t)]
        ids = tok(words, add_special_tokens=False)["input_ids"]
        pieces = np.array([len(x) for x in ids])
        row[f"tokens/word, {name}"] = round(pieces.mean(), 2)
        if name == "test posts":
            unk = unk_id(tok)
            row["[UNK] words, test posts"] = f"{np.mean([unk in x for x in ids]):.1%}" if unk is not None else "n/a"
            row["words in 4+ pieces, test posts"] = f"{(pieces >= 4).mean():.1%}"
    row["і ї є ґ"] = show_pieces(tok, "ї є ґ і")
    lengths = np.array([len(x) for x in tok(posts, add_special_tokens=True)["input_ids"]])
    row["posts over 256 tokens"] = int((lengths > 256).sum())
    return row


rows, examples = {}, {}
for label, name in CANDIDATES.items():
    try:
        tok = AutoTokenizer.from_pretrained(name, trust_remote_code=False)
    except Exception as e:  # a model that can't even be loaded is a result too
        rows[label] = {"vocab": f"failed: {type(e).__name__}"}
        continue
    rows[label] = profile(tok)
    examples[label] = {
        w: show_pieces(tok, w)
        for w in ["Святошинському", "Київщині", "Борщагівка", "Осокорках", "Дарницком", "Вишгородського"]
    }

table = pd.DataFrame(rows).T
table

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (588 > 512). Running this sequence through the model will result in indexing errors


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (623 > 512). Running this sequence through the model will result in indexing errors


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (672 > 512). Running this sequence through the model will result in indexing errors


,vocab,"tokens/word, uk sentences","tokens/word, ru sentences","tokens/word, test posts","[UNK] words, test posts","words in 4+ pieces, test posts","tokens/word, test places",і ї є ґ,posts over 256 tokens
mBERT,119547,2.08,1.93,2.14,0.1%,17.4%,2.71,ї|є|ґ|і,57
XLM-R base / large,250002,1.65,1.61,1.7,0.0%,5.7%,2.35,▁|ї|є|▁|ґ|і,39
mDeBERTa-v3,250101,2.07,1.93,2.07,0.1%,4.4%,2.44,▁|ї|▁|є|▁|ґ|▁|і,54
EuroBERT,128000,2.58,2.63,2.63,n/a,24.7%,3.4,ї|є|ґ|і,60
ruRoBERTa-large,50257,2.61,2.03,2.68,0.0%,25.9%,3.41,ї|�|�|▁|ґ|і,83
ukr-RoBERTa,52000,1.8,2.46,1.88,0.0%,4.3%,2.29,ї|є|ґ|і,28


In [4]:
pd.DataFrame(examples).T

,Святошинському,Київщині,Борщагівка,Осокорках,Дарницком,Вишгородського
mBERT,Свято|##шин|##ському,Київ|##щині,Бо|##р|##ща|##гі|##вка,О|##сок|##орка|##х,Дар|##ниц|##ком,В|##иш|##город|##ського
XLM-R base / large,Свято|шин|ському,Київ|щині,Бор|ща|г|івка,Ос|о|кор|ках,Дар|ни|ц|ком,Ви|ш|город|ського
mDeBERTa-v3,Свято|шин|ському,▁|Київ|щин|і,Бор|ща|г|івка,Ос|о|кор|ках,Дар|ниц|ком,Виш|город|ського
EuroBERT,С|в|я|то|шин|ському,К|и|їв|щин|і,Б|ор|щ|аг|ів|ка,Ос|ок|ор|ках,Д|ар|ниц|ком,В|иш|город|ського
ruRoBERTa-large,Св|ято|шин|сь|кому,К|и|ї|в|щ|ин|і,Б|орщ|аг|ів|ка,Ос|окор|ках,Дар|ниц|ком,В|иш|город|сь|кого
ukr-RoBERTa,Свято|шин|ському,Київ|щині,Бор|щагів|ка,Ос|окор|ках,Дар|ниц|ком,Ви|ш|город|ського


## What this means

- **XLM-R** splits Ukrainian and Russian equally well (1.65 and 1.61 tokens per word, 1.70 on the test
  posts), has the fewest words in 4+ pieces among the multilingual models and no `[UNK]`. Only 39 test
  posts need a second 256-token window. XLM-R large shares this tokenizer.
- **mDeBERTa-v3** has the same 250k vocabulary idea (CC100 SentencePiece) but splits a little more
  (2.07 per word); still fine, and the model is often stronger than XLM-R base at the same size.
- **mBERT** needs about 2.1 pieces per word and has a few `[UNK]` words (0.1%). Its WordPiece vocabulary
  is shared by 104 languages, so Cyrillic gets fewer entries. A reasonable but weaker baseline.
- **EuroBERT** splits both languages into about 2.6 pieces per word and a quarter of the test words into
  4 or more pieces ("Святошинському" in many byte pieces). Ukrainian is not among its pre-training
  languages, so we don't train it.
- **ruRoBERTa-large** handles Russian (2.03) much better than Ukrainian (2.61, 26% of test words in 4+
  pieces). It doesn't even have a token for the letter "є", which falls into two byte pieces (the "�"
in the table). It was pre-trained on Russian only; with 90% Ukrainian posts we skip it.
- **ukr-RoBERTa** is the mirror image: very good on Ukrainian (1.80, shortest posts), poor on Russian
  (2.46). An interesting single-language model, but it was trained on much less text than XLM-R, and
  10% of the posts and most of the Kaggle training sentences are Russian.

**Choice for training:** XLM-R base for the baseline and all data/optimizer experiments, then XLM-R
large, mDeBERTa-v3 base and mBERT with the best recipe. Posts longer than 256 tokens are split into
overlapping windows (stride 64) during training and prediction.